[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u6_aplicaciones.ipynb)

# Dif. U6 · Aplicaciones a ingeniería
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 6 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. Análisis de la gráfica: crecimiento, concavidad y puntos de inflexión
2. Teorema de Rolle, teorema del valor medio, diferenciales y método de Newton
3. Rapideces de variación relacionadas
4. Formas indeterminadas y regla de L'Hôpital
5. Optimización: criterios de primera y segunda derivada
6. Mecánica: velocidad, aceleración y dimensiones óptimas
7. Eléctrica y señales: capacitor, inductor y máximos de una señal
8. Industrial: utilidad máxima y lote económico
9. Civil: carga, cortante y momento flexionante máximo

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. Un óptimo solo sirve si cae dentro de lo que el problema permite: revisa siempre el dominio y los extremos.

**Etiqueta del repositorio.** `DIF-U6` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u6_aplicaciones/NN_*.py`; los fragmentos que el libro imprime, de `dif/u6_aplicaciones/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable (también cuando represente el tiempo o una cantidad), `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)` y `log(...)` (natural). Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x, y, u = sp.symbols("x y u", real=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
              "cbrt": lambda e: sp.sign(e) * sp.Abs(e)**sp.Rational(1, 3),
              "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que las derivadas salgan exactas
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_SIMBOLOS = {"x": x, "y": y, "u": u}


def parsear(texto, variables=("x",), funciones=None):
    """Convierte un texto en expresión de sympy o explica qué salió mal.
    variables: nombres permitidos (x, y o u). funciones: reemplaza el diccionario de funciones."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: x**3 - 3*x")
    fun = dict(_FUNCIONES if funciones is None else funciones)
    local = {**fun, **{v: _SIMBOLOS[v] for v in variables}}
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa {', '.join(variables)} como variable y, si hace falta, "
                         "sqrt, cbrt, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr):
        raise ValueError("Escribe una expresión, no una ecuación (sin el signo =).")
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - set(variables)
    if libres:
        raise ValueError(f"Símbolos no permitidos: {sorted(libres)}. Solo puedes usar {list(variables)}.")
    return expr


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def iguales(e1, e2):
    """True si las dos expresiones son equivalentes."""
    return sp.simplify(sp.sympify(e1) - sp.sympify(e2)) == 0


def evaluar_en(expr, var, a):
    """Valor exacto de expr en var = a; ValueError si no es un número real finito."""
    v = sp.simplify(expr.subs(var, sp.nsimplify(a)))
    if not (v.is_real and v.is_finite):
        raise ValueError(f"la expresión no tiene valor real en {var} = {a} (fuera del dominio).")
    return v


def cociente_num(expr, var, a, hh=1e-6):
    """Cociente incremental numérico (f(a + h) - f(a))/h, como comprobación independiente."""
    f = sp.lambdify(var, expr, "math")
    return (f(float(a) + hh) - f(float(a))) / hh


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def contrasta(mi_texto, referencia, variables=("x",), pista=""):
    """Celda Contrasta: compara tu resultado a mano (texto) con el de la calculadora."""
    if mi_texto is None or str(mi_texto).strip() == "":
        print("Falta tu cálculo a mano. Escríbelo arriba y vuelve a ejecutar la celda.")
        return None
    try:
        mio = parsear(str(mi_texto), variables)
    except ValueError as err:
        print(err); return None
    ok = iguales(mio, referencia)
    print("Coinciden." if ok else "No coinciden. " + pista)
    return ok

## 1. Análisis de la gráfica: crecimiento, concavidad y puntos de inflexión

| Signo | Significado |
|---|---|
| $f'>0$ / $f'<0$ | $f$ crece / decrece |
| $f''>0$ / $f''<0$ | cóncava hacia arriba / hacia abajo |
| $f''$ cambia de signo en $c$ | punto de inflexión ($f'$ es máxima o mínima ahí) |

$f'(c)=0$ no garantiza un extremo ($x^3$) y $f''(c)=0$ no garantiza inflexión ($x^4$): hay que revisar el **cambio de signo**.

La calculadora encuentra los puntos críticos y los candidatos a inflexión (reales), revisa los signos a cada lado y arma la tabla de crecimiento y concavidad por intervalos.

In [ ]:
def _signo(expr, v):
    val = float(expr.subs(x, v))
    return 1 if val > 1e-12 else (-1 if val < -1e-12 else 0)


def analizar(f, eps=sp.Rational(1, 1000)):
    """Diccionario con extremos locales [(c, tipo)] e inflexiones [c] de f (puntos donde f' o f'' se anulan)."""
    d1, d2 = sp.diff(f, x), sp.diff(f, x, 2)
    c1, c2 = sp.solveset(d1, x, domain=sp.S.Reals), sp.solveset(d2, x, domain=sp.S.Reals)
    if not all(isinstance(c, sp.FiniteSet) or c is sp.S.EmptySet for c in (c1, c2)):
        raise ValueError("no pude listar los puntos donde f' o f'' se anulan; prueba con un polinomio o una función más sencilla.")
    criticos, candidatos = sorted(c1, key=float), sorted(c2, key=float)
    extremos = []
    for c in criticos:
        izq, der = _signo(d1, c - eps), _signo(d1, c + eps)
        tipo = "máximo local" if (izq, der) == (1, -1) else "mínimo local" if (izq, der) == (-1, 1) else "sin extremo"
        extremos.append((c, tipo))
    inflexiones = [p for p in candidatos if _signo(d2, p - eps) * _signo(d2, p + eps) < 0]
    puntos = sorted(set(criticos) | set(candidatos), key=float)
    return {"f'": d1, "f''": d2, "extremos": extremos, "inflexiones": inflexiones, "puntos": puntos}


def tabla_signos(r):
    """[(a, b, signo de f', signo de f'')] en los intervalos que separan los puntos críticos y los candidatos a inflexión."""
    bordes = [-sp.oo, *r["puntos"], sp.oo]
    filas = []
    for a, b in zip(bordes[:-1], bordes[1:]):
        if a == -sp.oo and b == sp.oo:
            m = 0
        elif a == -sp.oo:
            m = b - 1
        elif b == sp.oo:
            m = a + 1
        else:
            m = (a + b) / 2
        filas.append((a, b, _signo(r["f'"], m), _signo(r["f''"], m)))
    return filas


def _txt(v, n):
    return "-∞" if v == -sp.oo else "∞" if v == sp.oo else cifras(v, n)


def calculadora_grafica(f_txt, n_cifras):
    try:
        f = parsear(f_txt)
        r = analizar(f)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    print("f'(x) =", sp.factor(r["f'"]), "   f''(x) =", sp.factor(r["f''"]))
    for c, tipo in r["extremos"]:
        print(f"  x = {c} ≈ {cifras(c, n_cifras)}: f'(x) = 0 -> {tipo}, f = {cifras(f.subs(x, c), n_cifras)}")
    for p in r["inflexiones"]:
        print(f"  x = {p} ≈ {cifras(p, n_cifras)}: f'' cambia de signo -> inflexión, f = {cifras(f.subs(x, p), n_cifras)}")
    if not r["inflexiones"]:
        print("  sin puntos de inflexión")
    print("tabla de signos:")
    for a, b, s1, s2 in tabla_signos(r):
        crece = "crece" if s1 > 0 else "decrece" if s1 < 0 else "constante"
        conc = "cóncava hacia arriba" if s2 > 0 else "cóncava hacia abajo" if s2 < 0 else "f'' = 0"
        print(f"  ({_txt(a, n_cifras)}, {_txt(b, n_cifras)}): f' {'+' if s1 > 0 else '-'} -> {crece};"
              f" f'' {'+' if s2 > 0 else '-'} -> {conc}")


widgets.interact(calculadora_grafica,
    f_txt=widgets.Text(value="x**3 - 6*x**2 + 9*x + 1", description="f(x) ="),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
_cub = analizar(x**3 - 6*x**2 + 9*x + 1)
PRUEBAS_1 = [
    ("cúbica: máximo en 1 y mínimo en 3", lambda: _cub["extremos"] == [(1, "máximo local"), (3, "mínimo local")]),
    ("cúbica: inflexión en 2", lambda: _cub["inflexiones"] == [2]),
    ("x^4: mínimo en 0 y sin inflexión", lambda: analizar(x**4)["extremos"] == [(0, "mínimo local")]
                                                   and analizar(x**4)["inflexiones"] == []),
    ("x^3: punto crítico sin extremo", lambda: analizar(x**3)["extremos"] == [(0, "sin extremo")]),
    ("e^(-x²): inflexiones en ±1/√2", lambda: analizar(sp.exp(-x**2))["inflexiones"] == [-1/sp.sqrt(2), 1/sp.sqrt(2)]),
    ("horno 1 - 2e^-x + e^-2x: inflexión en ln 2", lambda: analizar(1 - 2*sp.exp(-x) + sp.exp(-2*x))["inflexiones"] == [sp.log(2)]),
    ("x^4 - 6x^2: inflexiones en -1 y 1; tabla de 6 intervalos", lambda: analizar(x**4 - 6*x**2)["inflexiones"] == [-1, 1]
                                                                    and len(tabla_signos(analizar(x**4 - 6*x**2))) == 6),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Encuentra a mano los puntos críticos y la inflexión de $f(x)=x^3-3x^2$. Escribe las listas.

In [ ]:
mis_criticos = None      # escribe una lista de números, por ejemplo: [-1, 4]
mi_inflexion = None      # escribe un número

ref = analizar(x**3 - 3*x**2)
if mis_criticos is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da: críticos", [c for c, _ in ref["extremos"]], "| inflexión", ref["inflexiones"])
    print("coinciden" if sorted(mis_criticos) == [c for c, _ in ref["extremos"]] and [mi_inflexion] == ref["inflexiones"]
          else "NO coinciden: factoriza f' = 3x(x - 2) y resuelve f'' = 6x - 6 = 0")

## 2. Teorema del valor medio, diferenciales y método de Newton

**Valor medio.** Si $f$ es continua en $[a,b]$ y derivable en $(a,b)$, existe $c$ en $(a,b)$ con $f'(c)=\dfrac{f(b)-f(a)}{b-a}$. **Diferencial:** $dy=f'(x)\,dx\approx\Delta y$.

**Newton.** $x_{k+1}=x_k-\dfrac{f(x_k)}{f'(x_k)}$. Rápido cerca de una raíz simple; falla si $f'(x_k)=0$ y puede alejarse si se empieza lejos. La **bisección** de la Unidad 2 es más lenta pero siempre converge si $f$ es continua y cambia de signo en el intervalo. Newton se detiene cuando $|f(x_k)|$ y $|x_{k+1}-x_k|$ son menores que la tolerancia.

In [ ]:
def newton(f, x0, tol=1e-10, max_iter=50):
    """(raíz, iteraciones, lista de x_k) por el método de Newton. ValueError si f'(x_k) = 0 o si no converge."""
    df = sp.diff(f, x).replace(sp.DiracDelta, lambda *args: sp.S.Zero)   # la derivada de sign(x) fuera de 0
    F, DF = sp.lambdify(x, f, "math"), sp.lambdify(x, df, "math")
    xk, historia = float(x0), [float(x0)]
    for k in range(1, max_iter + 1):
        d = DF(xk)
        if d == 0:
            raise ValueError(f"f'({cifras(xk, 6)}) = 0: la tangente es horizontal y no corta el eje. Cambia x0.")
        nuevo = xk - F(xk) / d
        historia.append(nuevo)
        if abs(nuevo) > 1e12:
            raise ValueError("los valores crecen sin cota: Newton no converge desde ese x0.")
        if abs(F(nuevo)) < tol and abs(nuevo - xk) < tol:
            return nuevo, k, historia
        xk = nuevo
    raise ValueError(f"no convergió en {max_iter} iteraciones: revisa x0 o usa bisección.")


def biseccion(f, a, b, tol=1e-10):
    """(raíz, iteraciones) por bisección; ValueError si f no cambia de signo en [a, b]."""
    F = sp.lambdify(x, f, "math")
    if F(a) * F(b) > 0:
        raise ValueError("f no cambia de signo en [a, b]: la bisección no garantiza una raíz ahí.")
    k = 0
    while b - a > tol:
        m = (a + b) / 2
        a, b = (a, m) if F(a) * F(m) <= 0 else (m, b)
        k += 1
    return (a + b) / 2, k


def valor_medio(f, a, b):
    """Puntos c en (a, b) con f'(c) = pendiente de la secante."""
    a, b = sp.nsimplify(a), sp.nsimplify(b)
    m = (f.subs(x, b) - f.subs(x, a)) / (b - a)
    sol = sp.solveset(sp.Eq(sp.diff(f, x), m), x, domain=sp.Interval.open(a, b))
    return m, sorted(sol, key=float)


def calculadora_newton(f_txt, x0, tol, max_iter, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        if not 0 < tol < 1 or max_iter < 1:
            raise ValueError("la tolerancia debe estar entre 0 y 1 y el máximo de iteraciones ser al menos 1.")
        r, k, hist = newton(f, x0, tol, int(max_iter))
    except ValueError as err:
        print("Newton:", err)
    else:
        print("Newton:", " -> ".join(cifras(v, n_cifras) for v in hist[:8]), "..." if len(hist) > 8 else "")
        print(f"  raíz ≈ {cifras(r, n_cifras)} en {k} iteraciones ({n_cifras} cifras significativas)")
    try:
        rb, kb = biseccion(parsear(f_txt), a, b, tol)
        print(f"Bisección en [{a}, {b}]: raíz ≈ {cifras(rb, n_cifras)} en {kb} iteraciones")
    except ValueError as err:
        print("Bisección:", err)


widgets.interact(calculadora_newton,
    f_txt=widgets.Text(value="x**2 - 2", description="f(x) ="),
    x0=widgets.FloatText(value=1.0, description="x0"),
    tol=widgets.FloatText(value=1e-10, description="tolerancia"),
    max_iter=widgets.IntText(value=50, description="máx. iter."),
    a=widgets.FloatText(value=1.0, description="a (bisección)"), b=widgets.FloatText(value=2.0, description="b (bisección)"),
    n_cifras=widgets.IntSlider(value=8, min=1, max=12, description="cifras sig."));


def calculadora_valor_medio(f_txt, a, b, n_cifras):
    try:
        if not a < b:
            raise ValueError("el intervalo debe cumplir a < b.")
        m, cs = valor_medio(parsear(f_txt), a, b)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    print(f"pendiente de la secante en [{a}, {b}]: {cifras(m, n_cifras)}")
    print("puntos c con f'(c) igual a esa pendiente:", [cifras(c, n_cifras) for c in cs] or
          "ninguno: revisa si f es continua en [a, b] y derivable en (a, b)")


widgets.interact(calculadora_valor_medio,
    f_txt=widgets.Text(value="sqrt(x)", description="f(x) ="),
    a=widgets.FloatText(value=1.0, description="a"), b=widgets.FloatText(value=9.0, description="b"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("Newton √2 desde 1 en 5 iteraciones", lambda: cerca(newton(x**2 - 2, 1)[0], math.sqrt(2), 1e-12)
                                                  and newton(x**2 - 2, 1)[1] == 5),
    ("bisección √2 en [1, 2]: 34 iteraciones", lambda: biseccion(x**2 - 2, 1, 2)[1] == 34),
    ("e^(-x) = x desde 1: 0.5671433", lambda: cerca(newton(sp.exp(-x) - x, 1)[0], 0.5671432904, 1e-9)),
    ("x³ - 2x - 5 desde 3: x1 = 2.36", lambda: cerca(newton(x**3 - 2*x - 5, 3)[2][1], 2.36, 1e-12)),
    ("x² - 4 desde 0: f'(0) = 0, se avisa", lambda: _rechaza(lambda: newton(x**2 - 4, 0))),
    ("cbrt(x) desde 1: no converge, se avisa", lambda: _rechaza(lambda: newton(parsear("cbrt(x)"), 1))),
    ("valor medio √x en [1, 9]: c = 4", lambda: valor_medio(sp.sqrt(x), 1, 9) == (sp.Rational(1, 4), [4])),
    ("valor medio x³ en [0, 2]: c = 2/√3", lambda: valor_medio(x**3, 0, 2)[1] == [2/sp.sqrt(3)]),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Haz a mano dos pasos de Newton para $x^3-2x-5=0$ desde $x_0=2$ y escribe $x_2$ con 5 cifras significativas.

In [ ]:
mi_x2 = None             # escribe un número

ref = newton(x**3 - 2*x - 5, 2)[2][2]
if mi_x2 is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da x2 =", cifras(ref, 5))
    print("coincide" if abs(mi_x2 - ref) < 5e-5 else "NO coincide: x1 = 2 - f(2)/f'(2) = 2 - (-1)/10")

## 3. Rapideces de variación relacionadas

Si $x(t)$ y $y(t)$ cumplen $F(x,y)=0$ en todo instante, al derivar respecto a $t$:
$$F_x\,\frac{dx}{dt}+F_y\,\frac{dy}{dt}=0\quad\Longrightarrow\quad \frac{dy}{dt}=-\frac{F_x}{F_y}\,\frac{dx}{dt}.$$
Pasos: dibujar y nombrar; escribir la relación que vale **siempre**; derivar; sustituir los valores del instante **después** de derivar; interpretar signo y unidades.

La calculadora recibe la ecuación (con `=`), un punto de la curva y $dx/dt$, y devuelve $dy/dt$.

In [ ]:
def rapidez(ecuacion, px, py, dxdt):
    """dy/dt en el punto (px, py) de la curva, conocida dx/dt."""
    if ecuacion.count("=") != 1:
        raise ValueError("escribe una ecuación con un solo signo =, por ejemplo: x**2 + y**2 = 25")
    izq, der = ecuacion.split("=")
    F = parsear(izq, ("x", "y")) - parsear(der, ("x", "y"))
    punto = {x: sp.nsimplify(px), y: sp.nsimplify(py)}
    if sp.simplify(F.subs(punto)) != 0:
        raise ValueError(f"el punto ({px}, {py}) no cumple la ecuación: no corresponde a ningún instante.")
    Fx, Fy = sp.diff(F, x).subs(punto), sp.diff(F, y).subs(punto)
    if sp.simplify(Fy) == 0:
        raise ValueError("el coeficiente de dy/dt se anula en ese punto: la relación no determina dy/dt.")
    return sp.simplify(-Fx / Fy * sp.nsimplify(dxdt))


def calculadora_rapidez(ecuacion, px, py, dxdt, n_cifras):
    try:
        v = rapidez(ecuacion, px, py, dxdt)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"dy/dt = {v} ≈ {cifras(v, n_cifras)}  ({n_cifras} cifras significativas)")
    print("signo:", "y crece" if v > 0 else "y decrece" if v < 0 else "y no cambia en ese instante")


widgets.interact(calculadora_rapidez,
    ecuacion=widgets.Text(value="x**2 + y**2 = 25", description="ecuación"),
    px=widgets.FloatText(value=3.0, description="x"), py=widgets.FloatText(value=4.0, description="y"),
    dxdt=widgets.FloatText(value=0.5, description="dx/dt"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido). En la tolva, x es el volumen y y la altura: x = π y³/12.
PRUEBAS_3 = [
    ("escalera en (3, 4) con dx/dt = 0.5: -0.375", lambda: rapidez("x**2 + y**2 = 25", 3, 4, 0.5) == sp.Rational(-3, 8)),
    ("escalera en (4, 3) con dx/dt = 0.3: -0.4", lambda: rapidez("x**2 + y**2 = 25", 4, 3, 0.3) == sp.Rational(-2, 5)),
    ("tolva h = 2 con dV/dt = -0.4: -0.4/π",
     lambda: iguales(rapidez("x = pi*y**3/12", sp.pi*8/12, 2, -0.4), -sp.Rational(2, 5)/sp.pi)),
    ("tolva h = 1: cuatro veces más rápido",
     lambda: iguales(rapidez("x = pi*y**3/12", sp.pi/12, 1, -0.4) / rapidez("x = pi*y**3/12", sp.pi*8/12, 2, -0.4), 4)),
    ("punto fuera de la curva se rechaza", lambda: _rechaza(lambda: rapidez("x**2 + y**2 = 25", 1, 1, 1))),
    ("coeficiente nulo se avisa", lambda: _rechaza(lambda: rapidez("x**2 + y**2 = 25", 5, 0, 1))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Escalera de 5 m: calcula a mano $dy/dt$ cuando la base está a 4 m de la pared y se aleja a 0.4 m/s. Escribe el número.

In [ ]:
mi_valor = None          # escribe un número

ref = rapidez("x**2 + y**2 = 25", 4, 3, 0.4)
if mi_valor is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref, "≈", cifras(ref, 4), "m/s")
    print("coincide" if abs(mi_valor - float(ref)) < 5e-4 else "NO coincide: 2x x' + 2y y' = 0 con x = 4, y = 3")

## 4. Formas indeterminadas y regla de L'Hôpital

Si $\lim\frac{f}{g}$ es de la forma $\frac00$ o $\frac{\infty}{\infty}$ y $\lim\frac{f'}{g'}$ existe, entonces $\lim\frac fg=\lim\frac{f'}{g'}$. Se derivan **por separado** numerador y denominador, y **antes de cada aplicación** se comprueba que la forma siga siendo indeterminada.

La calculadora aplica la regla paso a paso (hasta cinco veces). Si la forma no es indeterminada, no la aplica y da el límite directo. Para $a=\infty$ escribe `oo` o `∞`. Los resultados son exactos.

In [ ]:
def _forma(num, den, a):
    ln, ld = sp.limit(num, x, a), sp.limit(den, x, a)
    if ln == 0 and ld == 0:
        return "0/0"
    if ln in (sp.oo, -sp.oo) and ld in (sp.oo, -sp.oo):
        return "∞/∞"
    return None


def _punto(a):
    return sp.oo if str(a).strip() in ("oo", "inf", "∞") else sp.nsimplify(a)


def lhopital(num_txt, den_txt, a, max_pasos=5):
    """(límite, pasos) aplicando L'Hôpital mientras la forma sea indeterminada. ValueError si de entrada no lo es."""
    num, den = parsear(num_txt), parsear(den_txt)
    a = _punto(a)
    if _forma(num, den, a) is None:
        raise ValueError(f"la forma no es indeterminada (numerador -> {sp.limit(num, x, a)}, denominador -> "
                         f"{sp.limit(den, x, a)}): no se aplica L'Hôpital; el límite es directo.")
    pasos = []
    while (forma := _forma(num, den, a)) is not None:
        if len(pasos) == max_pasos:
            raise ValueError("después de cinco aplicaciones sigue indeterminado: reescribe el cociente.")
        pasos.append((forma, num, den))
        # se derivan numerador y denominador por separado y se simplifica el nuevo cociente
        num, den = sp.fraction(sp.powsimp(sp.together(sp.diff(num, x) / sp.diff(den, x)), force=True))
    return sp.limit(num / den, x, a), pasos


def calculadora_lhopital(num_txt, den_txt, a):
    try:
        num, den, p = parsear(num_txt), parsear(den_txt), _punto(a)
        if _forma(num, den, p) is None:
            print(f"no es indeterminada (numerador -> {sp.limit(num, x, p)}, denominador -> {sp.limit(den, x, p)}):"
                  f" no se aplica L'Hôpital. Límite directo = {sp.limit(num / den, x, p)}")
            return
        lim, pasos = lhopital(num_txt, den_txt, a)
    except (ValueError, TypeError, sp.SympifyError) as err:
        print("Revisa la entrada:", err); return
    for k, (forma, n_, d_) in enumerate(pasos, 1):
        print(f"paso {k}: ({n_})/({d_}) es {forma} -> derivar por separado")
    print("límite =", lim, "| comprobación con sp.limit:", sp.limit(num / den, x, p))


widgets.interact(calculadora_lhopital,
    num_txt=widgets.Text(value="1 - cos(x)", description="numerador"),
    den_txt=widgets.Text(value="x**2", description="denominador"),
    a=widgets.Text(value="0", description="x ->"));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("(1 - cos x)/x² en 0: 1/2 con dos pasos", lambda: lhopital("1 - cos(x)", "x**2", 0)[0] == sp.Rational(1, 2)
                                                         and len(lhopital("1 - cos(x)", "x**2", 0)[1]) == 2),
    ("(e^x - 1)/x en 0: 1", lambda: lhopital("exp(x) - 1", "x", 0)[0] == 1),
    ("x³/e^x en ∞: 0 con tres pasos", lambda: lhopital("x**3", "exp(x)", "∞")[0] == 0
                                              and len(lhopital("x**3", "exp(x)", "∞")[1]) == 3),
    ("tan x / x en 0: 1", lambda: lhopital("tan(x)", "x", 0)[0] == 1),
    ("50 ln x / x^0.2 en ∞: 0", lambda: lhopital("50*log(x)", "x**(1/5)", "oo")[0] == 0),
    ("(1 - cos x)/(x² + x) en 0: 0 con un solo paso", lambda: lhopital("1 - cos(x)", "x**2 + x", 0)[0] == 0
                                                         and len(lhopital("1 - cos(x)", "x**2 + x", 0)[1]) == 1),
    ("(x + 1)/(x + 2) en 0 no es indeterminado: se rechaza", lambda: _rechaza(lambda: lhopital("x + 1", "x + 2", 0))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $\displaystyle\lim_{x\to0}\frac{e^{2x}-1-2x}{x^2}$ y escribe el número. ¿Cuántas veces aplicaste la regla?

In [ ]:
mi_limite = None         # escribe un número

ref, pasos_ref = lhopital("exp(2*x) - 1 - 2*x", "x**2", 0)
if mi_limite is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref, "con", len(pasos_ref), "aplicaciones")
    print("coincide" if iguales(sp.nsimplify(mi_limite), ref) else "NO coincide: comprueba 0/0 antes de cada aplicación")

## 5. Optimización: criterios de primera y segunda derivada

**Intervalo cerrado.** Si $f$ es continua en $[a,b]$, el máximo y el mínimo absolutos están entre los puntos críticos de $(a,b)$ y los extremos $a$, $b$: se evalúa $f$ en todos y se compara. **Segunda derivada:** $f'(c)=0$ y $f''(c)<0$ da máximo local; $f''(c)>0$, mínimo; $f''(c)=0$, no decide.

**Retoma tu predicción de la semilla.** La celda siguiente optimiza la caja de cartón de 60 cm × 40 cm.

In [ ]:
def extremos_intervalo(f, a, b):
    """(candidatos {x: f(x)}, x del máximo, x del mínimo) en [a, b]."""
    a, b = sp.nsimplify(a), sp.nsimplify(b)
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    crit = sp.solveset(sp.diff(f, x), x, domain=sp.Interval.open(a, b))
    if crit is sp.S.EmptySet:
        crit = []
    elif not isinstance(crit, sp.FiniteSet):
        raise ValueError("no pude listar los puntos críticos en el intervalo; prueba con otra función.")
    cand = {c: sp.simplify(f.subs(x, c)) for c in [a, b, *crit]}
    return cand, max(cand, key=lambda c: float(cand[c])), min(cand, key=lambda c: float(cand[c]))


def clasificar(f, c):
    """'máximo local', 'mínimo local' o 'sin extremo' en un punto crítico c (segunda derivada o cambio de signo)."""
    d2 = sp.diff(f, x, 2).subs(x, c)
    if d2 < 0:
        return "máximo local"
    if d2 > 0:
        return "mínimo local"
    d1 = sp.diff(f, x)
    e = sp.Rational(1, 1000)
    izq, der = d1.subs(x, c - e), d1.subs(x, c + e)
    return "máximo local" if izq > 0 > der else "mínimo local" if izq < 0 < der else "sin extremo"


caja = x*(60 - 2*x)*(40 - 2*x)
cand, xmax, _ = extremos_intervalo(caja, 0, 20)
print("caja: corte óptimo x ≈", cifras(xmax, 4), "cm, volumen ≈", cifras(cand[xmax], 4), "cm³")


def calculadora_optimizacion(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        cand, xM, xm = extremos_intervalo(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for c, v in cand.items():
        nota = "extremo del intervalo" if c in (sp.nsimplify(a), sp.nsimplify(b)) else clasificar(f, c)
        print(f"  x = {cifras(c, n_cifras):>10}: f = {cifras(v, n_cifras):>10}   ({nota})")
    print(f"máximo absoluto en x ≈ {cifras(xM, n_cifras)}; mínimo absoluto en x ≈ {cifras(xm, n_cifras)}")


widgets.interact(calculadora_optimizacion,
    f_txt=widgets.Text(value="x**3 - 6*x**2 + 9*x + 1", description="f(x) ="),
    a=widgets.FloatText(value=0.5, description="a"), b=widgets.FloatText(value=4.5, description="b"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("cúbica en [0.5, 4.5]: máximo en 4.5 (extremo), mínimo en 3",
     lambda: extremos_intervalo(x**3 - 6*x**2 + 9*x + 1, 0.5, 4.5)[1:] == (sp.Rational(9, 2), 3)),
    ("caja en [0, 20]: corte ≈ 7.85 cm, volumen ≈ 8450", lambda: cerca(xmax, 7.8475, 1e-4) and round(float(cand[xmax])) == 8450),
    ("caja con la máquina, [0, 5]: máximo en 5 con 7500", lambda: extremos_intervalo(caja, 0, 5)[1] == 5
                                                               and extremos_intervalo(caja, 0, 5)[0][5] == 7500),
    ("x(10 - x): cuadrado de 5, área 25", lambda: extremos_intervalo(x*(10 - x), 0, 10)[1] == 5),
    ("x^4 en 0: mínimo aunque f''(0) = 0", lambda: clasificar(x**4, 0) == "mínimo local"),
    ("x^3 en 0: sin extremo", lambda: clasificar(x**3, 0) == "sin extremo"),
    ("intervalo con a ≥ b se rechaza", lambda: _rechaza(lambda: extremos_intervalo(x**2, 3, 1))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Calcula a mano el volumen de la caja con los cortes de la semilla, 2 cm, 8 cm y 15 cm, y compáralos con el óptimo.

In [ ]:
mis_V = None             # escribe una lista de tres números: [V(2), V(8), V(15)]

if mis_V is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", [caja.subs(x, c) for c in (2, 8, 15)], "| óptimo ≈", cifras(cand[xmax], 4), "cm³")
    print("coinciden" if list(mis_V) == [caja.subs(x, c) for c in (2, 8, 15)] else "NO coinciden: V = x(60 - 2x)(40 - 2x)")

## 6. Mecánica: velocidad, aceleración y dimensiones óptimas

$v=s'$ y $a=v'=s''$. En reposo donde $v=0$; la distancia recorrida suma los tramos entre cambios de sentido; el desplazamiento es $s(t_1)-s(t_0)$.

Dos calculadoras: la primera analiza un movimiento $s(t)$ (escribe $t$ como `x`); la segunda dimensiona un tanque cilíndrico cerrado de volumen $V$ con precios distintos para tapas y pared y un radio máximo.

In [ ]:
def movimiento(s_txt, t0, t1):
    """Diccionario con v, a, instantes de reposo, distancia recorrida y desplazamiento en [t0, t1]."""
    s = parsear(s_txt)
    t0, t1 = sp.nsimplify(t0), sp.nsimplify(t1)
    if not t0 < t1:
        raise ValueError("el intervalo debe cumplir t0 < t1.")
    v, a = sp.diff(s, x), sp.diff(s, x, 2)
    reposo = sorted(sp.solveset(v, x, domain=sp.Interval.open(t0, t1)), key=float)
    marcas = [t0, *reposo, t1]
    distancia = sum(abs(s.subs(x, q) - s.subs(x, p)) for p, q in zip(marcas[:-1], marcas[1:]))
    return {"v": v, "a": a, "reposo": reposo, "distancia": sp.simplify(distancia),
            "desplazamiento": sp.simplify(s.subs(x, t1) - s.subs(x, t0))}


def tanque_optimo(V, precio_tapas, precio_pared, r_max=None):
    """(r, h, costo) del tanque cilíndrico cerrado de costo mínimo; respeta r <= r_max si se da."""
    if min(V, precio_tapas, precio_pared) <= 0:
        raise ValueError("volumen y precios deben ser positivos.")
    r = sp.symbols("r", positive=True)
    C = precio_tapas * 2 * sp.pi * r**2 + precio_pared * 2 * V / r
    ro = sp.solve(sp.diff(C, r), r)[0]
    if r_max is not None and ro > r_max:
        ro = sp.nsimplify(r_max)          # C decrece antes del óptimo: el mejor radio permitido es el tope
    return float(ro), float(V / (sp.pi * ro**2)), float(C.subs(r, ro))


def calculadora_movimiento(s_txt, t0, t1, n_cifras):
    try:
        r = movimiento(s_txt, t0, t1)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("v(t) =", r["v"], "   a(t) =", r["a"])
    print("reposo en t =", [cifras(q, n_cifras) for q in r["reposo"]] or "ningún instante del intervalo")
    print(f"distancia recorrida = {cifras(r['distancia'], n_cifras)}   desplazamiento = {cifras(r['desplazamiento'], n_cifras)}")


widgets.interact(calculadora_movimiento,
    s_txt=widgets.Text(value="20*x - 2.5*x**2", description="s(t) ="),
    t0=widgets.FloatText(value=0.0, description="t0"), t1=widgets.FloatText(value=6.0, description="t1"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));


def calculadora_tanque(V, precio_tapas, precio_pared, r_max, n_cifras):
    try:
        r, h, c = tanque_optimo(V, precio_tapas, precio_pared, r_max if r_max > 0 else None)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"r = {cifras(r, n_cifras)} m, h = {cifras(h, n_cifras)} m, h/r = {cifras(h / r, n_cifras)},"
          f" costo = {cifras(c, n_cifras)} pesos  ({n_cifras} cifras significativas)")


widgets.interact(calculadora_tanque,
    V=widgets.FloatText(value=0.5, description="V (m³)"),
    precio_tapas=widgets.FloatText(value=675, description="tapas $/m²"),
    precio_pared=widgets.FloatText(value=450, description="pared $/m²"),
    r_max=widgets.FloatText(value=0, description="r máx (0 = sin tope)"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 6 (resultado conocido)
_q = movimiento("0.4*(10*(x/2)**3 - 15*(x/2)**4 + 6*(x/2)**5)", 0, 2)
PRUEBAS_6 = [
    ("frenado 20t - 2.5t²: reposo en 4 s, 40 m", lambda: movimiento("20*x - 2.5*x**2", 0, 6)["reposo"] == [4]
                                                         and movimiento("20*x - 2.5*x**2", 0, 4)["desplazamiento"] == 40),
    ("t³ - 9t² + 24t en [0, 5]: reposo 2 y 4; distancia 28; desplazamiento 20",
     lambda: movimiento("x**3 - 9*x**2 + 24*x", 0, 5)["reposo"] == [2, 4]
             and movimiento("x**3 - 9*x**2 + 24*x", 0, 5)["distancia"] == 28
             and movimiento("x**3 - 9*x**2 + 24*x", 0, 5)["desplazamiento"] == 20),
    ("quíntico: v máx 0.375 en t = 1", lambda: _q["v"].subs(x, 1) == sp.Rational(3, 8)),
    ("quíntico: v y a nulas en 0 y 2", lambda: [_q["v"].subs(x, k) for k in (0, 2)] == [0, 0] and [_q["a"].subs(x, k) for k in (0, 2)] == [0, 0]),
    ("tanque 0.5 m³, 675/450: r ≈ 0.3758, h/r = 3, ≈ 1796.4",
     lambda: cerca(tanque_optimo(0.5, 675, 450)[0], 0.375751, 1e-5) and cerca(tanque_optimo(0.5, 675, 450)[1] / tanque_optimo(0.5, 675, 450)[0], 3, 1e-9)
             and round(tanque_optimo(0.5, 675, 450)[2], 1) == 1796.4),
    ("tanque con r ≤ 0.35: ≈ 1805.26", lambda: round(tanque_optimo(0.5, 675, 450, 0.35)[2], 2) == 1805.26),
    ("precios iguales: h = 2r", lambda: cerca(tanque_optimo(1.0, 100, 100)[1] / tanque_optimo(1.0, 100, 100)[0], 2, 1e-9)),
]
for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6).** Para $s(t)=2t^3-9t^2+12t$ (m, s), encuentra a mano los instantes de reposo en $[0,3]$ y escríbelos como lista.

In [ ]:
mis_instantes = None     # escribe una lista, por ejemplo: [0.5, 2.5]

ref = movimiento("2*x**3 - 9*x**2 + 12*x", 0, 3)["reposo"]
if mis_instantes is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref)
    print("coinciden" if sorted(mis_instantes) == ref else "NO coinciden: v = 6t² - 18t + 12 = 6(t - 1)(t - 2)")

## 7. Eléctrica y señales: $i_C=C\,dv/dt$, $v_L=L\,di/dt$; máximos de una señal

Dos calculadoras: la de capacitor da $i_C$ a partir de $C$ y $v(t)$; la de inductor, $v_L$ a partir de $L$ e $i(t)$. Las dos buscan el máximo y el mínimo del resultado en un intervalo. Convención pasiva: $i$ entra por la terminal $+$.

Unidades: $C$ en faradios, $L$ en henrios, $v$ en volts, $i$ en amperes, $t$ en segundos (escribe $t$ como `x`). Los extremos de una señal en un intervalo están donde su derivada se anula **o en los extremos del intervalo**.

In [ ]:
def corriente_capacitor(C, v_txt):
    if C <= 0:
        raise ValueError("la capacitancia debe ser positiva (en faradios: 10 µF = 10e-6).")
    return sp.nsimplify(C) * sp.diff(parsear(v_txt), x)


def tension_inductor(L, i_txt):
    if L <= 0:
        raise ValueError("la inductancia debe ser positiva (en henrios: 20 mH = 0.02).")
    return sp.nsimplify(L) * sp.diff(parsear(i_txt), x)


def extremos_senal(senal, t0, t1, n=4001):
    """(t del máximo, valor, t del mínimo, valor) en [t0, t1]: puntos críticos refinados y extremos del intervalo."""
    f = sp.lambdify(x, senal, "numpy")
    ts = np.linspace(float(t0), float(t1), n)
    with np.errstate(all="ignore"):
        vs = np.broadcast_to(f(ts), ts.shape).astype(float)
    cand = [float(t0), float(t1)]
    dfun = sp.lambdify(x, sp.diff(senal, x), "numpy")
    signos = np.sign(np.broadcast_to(dfun(ts), ts.shape))
    for k in np.where(signos[:-1] * signos[1:] < 0)[0]:    # cambios de signo de la derivada
        a, b = ts[k], ts[k + 1]
        for _ in range(60):                                # bisección sobre la derivada
            m = (a + b) / 2
            a, b = (a, m) if np.sign(dfun(a)) * np.sign(dfun(m)) <= 0 else (m, b)
        cand.append(float((a + b) / 2))
    vals = [float(senal.subs(x, c)) for c in cand]
    kM, km = int(np.argmax(vals)), int(np.argmin(vals))
    return cand[kM], vals[kM], cand[km], vals[km]


def _reporte(nombre, unidad, senal, t0_ms, t1_ms, n):
    tM, sM, tm, sm = extremos_senal(senal, t0_ms / 1000, t1_ms / 1000)
    print(f"{nombre}(t) =", sp.N(senal, n), unidad)
    print(f"en [{t0_ms}, {t1_ms}] ms: máximo {cifras(sM, n)} {unidad} en t = {cifras(tM * 1000, n)} ms"
          f" | mínimo {cifras(sm, n)} {unidad} en t = {cifras(tm * 1000, n)} ms  ({n} cifras significativas)")


def calculadora_capacitor(C_uF, v_txt, t0_ms, t1_ms, n_cifras):
    try:
        _reporte("i_C", "A", corriente_capacitor(C_uF * 1e-6, v_txt), t0_ms, t1_ms, n_cifras)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err)


widgets.interact(calculadora_capacitor,
    C_uF=widgets.FloatText(value=2.0, description="C (µF)"),
    v_txt=widgets.Text(value="50*(exp(-500*x) - exp(-2000*x))", description="v(t) ="),
    t0_ms=widgets.FloatText(value=0.0, description="t0 (ms)"), t1_ms=widgets.FloatText(value=6.0, description="t1 (ms)"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));


def calculadora_inductor(L_mH, i_txt, t0_ms, t1_ms, n_cifras):
    try:
        _reporte("v_L", "V", tension_inductor(L_mH * 1e-3, i_txt), t0_ms, t1_ms, n_cifras)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err)


widgets.interact(calculadora_inductor,
    L_mH=widgets.FloatText(value=20.0, description="L (mH)"),
    i_txt=widgets.Text(value="3*sin(377*x)", description="i(t) ="),
    t0_ms=widgets.FloatText(value=0.0, description="t0 (ms)"), t1_ms=widgets.FloatText(value=20.0, description="t1 (ms)"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 7 (resultado conocido)
_pulso = parsear("50*(exp(-500*x) - exp(-2000*x))")
PRUEBAS_7 = [
    ("10 µF con 10 sen 377t: pico 0.0377 A", lambda: cerca(corriente_capacitor(10e-6, "10*sin(377*x)").subs(x, 0), 0.0377, 1e-12)),
    ("20 mH con 3 sen 377t: pico 22.62 V", lambda: cerca(tension_inductor(0.02, "3*sin(377*x)").subs(x, 0), 22.62, 1e-9)),
    ("pulso: v máx ≈ 23.62 V en ≈ 0.924 ms", lambda: round(extremos_senal(_pulso, 0, 0.006)[1], 2) == 23.62
                                                    and round(extremos_senal(_pulso, 0, 0.006)[0] * 1000, 3) == 0.924),
    ("pulso: i máx 0.15 A en t = 0 (extremo del intervalo)",
     lambda: extremos_senal(corriente_capacitor(2e-6, "50*(exp(-500*x) - exp(-2000*x))"), 0, 0.006)[:2] == (0.0, 0.15)),
    ("pulso: i mín ≈ -0.0149 A en ≈ 1.85 ms",
     lambda: round(extremos_senal(corriente_capacitor(2e-6, "50*(exp(-500*x) - exp(-2000*x))"), 0, 0.006)[3], 4) == -0.0149),
    ("capacitancia no positiva se rechaza", lambda: _rechaza(lambda: corriente_capacitor(0, "x"))),
]
for nombre, prueba in PRUEBAS_7:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 7).** Calcula a mano la corriente pico de un capacitor de 47 µF con $v=5\,\mathrm{sen}\,1000t$ y escribe el número en amperes.

In [ ]:
mi_pico = None           # escribe un número

ref = float(corriente_capacitor(47e-6, "5*sin(1000*x)").subs(x, 0))
if mi_pico is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", cifras(ref, 4), "A")
    print("coincide" if abs(mi_pico - ref) < 5e-4 else "NO coincide: i = C·5·1000·cos(1000t) y el pico es el coeficiente")

## 8. Industrial: utilidad máxima y lote económico

**Utilidad.** $U=R-C$ con $R=q\,p(q)$. En un máximo interior, $R'(q)=C'(q)$ (ingreso marginal igual a costo marginal) y $U''<0$. Si hay capacidad máxima, se compara con el extremo.

**Lote económico.** $CT(Q)=\dfrac{DS}{Q}+\dfrac{HQ}{2}$, mínimo en $Q^*=\sqrt{2DS/H}$. Supuestos: demanda constante, reposición instantánea, sin faltantes ni descuentos.

Escribe la cantidad $q$ como `x`.

In [ ]:
def utilidad_maxima(p_txt, C_txt, q_max):
    """(q de utilidad máxima en [0, q_max], U, q de ingreso máximo, U en ese q)."""
    p, C = parsear(p_txt), parsear(C_txt)
    R, U = x * p, x * p - C
    qU = [c for c in sp.solveset(sp.diff(U, x), x, domain=sp.Interval.open(0, q_max))]
    cand = [sp.Integer(0), sp.nsimplify(q_max), *qU]
    mejor = max(cand, key=lambda c: float(U.subs(x, c)))
    qR = [c for c in sp.solveset(sp.diff(R, x), x, domain=sp.Interval.open(0, sp.oo))]
    qR = qR[0] if qR else None
    return mejor, U.subs(x, mejor), qR, (U.subs(x, qR) if qR is not None else None)


def lote_economico(D, S, H, Q_max=None):
    """(Q*, costo anual en Q*, Q usado, costo con Q usado)."""
    if min(D, S, H) <= 0:
        raise ValueError("D, S y H deben ser positivos.")
    Qs = math.sqrt(2 * D * S / H)
    CT = lambda Q: D * S / Q + H * Q / 2
    Qu = min(Qs, Q_max) if Q_max else Qs        # CT decrece antes de Q*: con tope, el mejor es el tope
    return Qs, CT(Qs), Qu, CT(Qu)


def calculadora_utilidad(p_txt, C_txt, q_max, n_cifras):
    try:
        q, U, qR, UR = utilidad_maxima(p_txt, C_txt, q_max)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"utilidad máxima en [0, {q_max}]: q = {cifras(q, n_cifras)}, U = {cifras(U, n_cifras)}  ({n_cifras} cifras significativas)")
    if qR is not None:
        print(f"(el ingreso es máximo en q = {cifras(qR, n_cifras)}, donde U = {cifras(UR, n_cifras)})")


widgets.interact(calculadora_utilidad,
    p_txt=widgets.Text(value="500 - 0.5*x", description="p(q) ="),
    C_txt=widgets.Text(value="20000 + 140*x", description="C(q) ="),
    q_max=widgets.FloatText(value=1000, description="capacidad"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=8, description="cifras sig."));


def calculadora_lote(D, S, H, Q_max, n_cifras):
    try:
        Qs, CTs, Qu, CTu = lote_economico(D, S, H, Q_max if Q_max > 0 else None)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"Q* = {cifras(Qs, n_cifras)}, costo anual {cifras(CTs, n_cifras)}  ({n_cifras} cifras significativas)")
    if Qu != Qs:
        print(f"con el tope: Q = {cifras(Qu, n_cifras)}, costo anual {cifras(CTu, n_cifras)}"
              f" ({cifras(100 * (CTu / CTs - 1), 3)} % más)")


widgets.interact(calculadora_lote,
    D=widgets.FloatText(value=12000, description="D (unid/año)"), S=widgets.FloatText(value=800, description="S ($/pedido)"),
    H=widgets.FloatText(value=30, description="H ($/unid·año)"), Q_max=widgets.FloatText(value=0, description="tope (0 = no)"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 8 (resultado conocido)
PRUEBAS_8 = [
    ("utilidad: q = 360, U = 44800", lambda: utilidad_maxima("500 - 0.5*x", "20000 + 140*x", 1000)[:2] == (360, 44800)),
    ("ingreso máximo en 500 con U = 35000", lambda: utilidad_maxima("500 - 0.5*x", "20000 + 140*x", 1000)[2:] == (500, 35000)),
    ("con capacidad 300: q = 300, U = 43000", lambda: utilidad_maxima("500 - 0.5*x", "20000 + 140*x", 300)[:2] == (300, 43000)),
    ("EOQ 12000/800/30: Q* = 800, 24000", lambda: lote_economico(12000, 800, 30)[:2] == (800.0, 24000.0)),
    ("EOQ con tope 600: 25000", lambda: lote_economico(12000, 800, 30, 600)[2:] == (600, 25000.0)),
    ("EOQ proveedor B 12000/1100/30: Q* ≈ 938.1, ≈ 28142.5", lambda: round(lote_economico(12000, 1100, 30)[0], 1) == 938.1
                                                                  and round(lote_economico(12000, 1100, 30)[1], 1) == 28142.5),
    ("datos no positivos se rechazan", lambda: _rechaza(lambda: lote_economico(0, 800, 30))),
]
for nombre, prueba in PRUEBAS_8:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 8).** Con $D=5000$, $S=400$ y $H=25$, calcula a mano $Q^*$ y escribe el número.

In [ ]:
mi_Q = None              # escribe un número

ref = lote_economico(5000, 400, 25)[0]
if mi_Q is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da Q* =", cifras(ref, 6))
    print("coincide" if abs(mi_Q - ref) < 0.5 else "NO coincide: Q* = raíz de 2·D·S/H")

## 9. Civil: $dV/dx=-w(x)$, $dM/dx=V(x)$; momento flexionante máximo

Convención del libro: carga $w$ positiva hacia abajo y momento positivo con la concavidad hacia arriba. El momento tiene un extremo donde $V=0$; en apoyos simples, $M=0$. Escribe la posición como `x` (en metros).

La calculadora recibe $M(x)$ (kN·m) y el claro $L$; obtiene $V$ y $w$, revisa los apoyos y encuentra el momento de mayor valor absoluto comparando los ceros de $V$ con los extremos.

In [ ]:
def viga(M_txt, L):
    """Diccionario con V, w, M en los apoyos y (x, M) del momento de mayor valor absoluto en [0, L]."""
    M = parsear(M_txt)
    L = sp.nsimplify(L)
    if L <= 0:
        raise ValueError("el claro L debe ser positivo.")
    V = sp.diff(M, x)
    ceros = [c for c in sp.solveset(V, x, domain=sp.Interval(0, L))]
    cand = {c: M.subs(x, c) for c in [sp.Integer(0), L, *ceros]}
    xm = max(cand, key=lambda c: abs(float(cand[c])))
    return {"V": V, "w": sp.simplify(-sp.diff(V, x)), "apoyos": (M.subs(x, 0), M.subs(x, L)),
            "ceros_V": ceros, "x_max": xm, "M_max": sp.simplify(cand[xm])}


def calculadora_viga(M_txt, L, n_cifras):
    try:
        r = viga(M_txt, L)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("V(x) =", r["V"], "kN    w(x) = -V' =", r["w"], "kN/m")
    print("M en los apoyos:", r["apoyos"], "(en apoyos simples debe ser 0)")
    print(f"V = 0 en x = {[cifras(c, n_cifras) for c in r['ceros_V']]}")
    print(f"momento máximo: {cifras(r['M_max'], n_cifras)} kN·m en x = {cifras(r['x_max'], n_cifras)} m")


widgets.interact(calculadora_viga,
    M_txt=widgets.Text(value="12*x - x**3/3", description="M(x) ="),
    L=widgets.FloatText(value=6.0, description="L (m)"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 9 (resultado conocido)
PRUEBAS_9 = [
    ("uniforme w = 10, L = 8: máx 80 en 4", lambda: (viga("40*x - 5*x**2", 8)["x_max"], viga("40*x - 5*x**2", 8)["M_max"]) == (4, 80)),
    ("uniforme: w = 10", lambda: viga("40*x - 5*x**2", 8)["w"] == 10),
    ("triangular: V = 12 - x², w = 2x", lambda: iguales(viga("12*x - x**3/3", 6)["V"], 12 - x**2) and iguales(viga("12*x - x**3/3", 6)["w"], 2*x)),
    ("triangular: máx 16√3 en √12", lambda: viga("12*x - x**3/3", 6)["x_max"] == sp.sqrt(12)
                                            and iguales(viga("12*x - x**3/3", 6)["M_max"], 16*sp.sqrt(3))),
    ("apoyos de la triangular: M = 0 en 0 y 6", lambda: viga("12*x - x**3/3", 6)["apoyos"] == (0, 0)),
    ("claro no positivo se rechaza", lambda: _rechaza(lambda: viga("x", 0))),
]
for nombre, prueba in PRUEBAS_9:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 9).** Carga uniforme con $w=6$ kN/m y $L=5$ m: calcula a mano $M_{\max}=wL^2/8$ y escribe el número. Para comprobar, escribe en la calculadora $M(x)=15x-3x^2$.

In [ ]:
mi_M = None              # escribe un número

ref = viga("15*x - 3*x**2", 5)["M_max"]
if mi_M is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref, "=", cifras(ref, 4), "kN·m")
    print("coincide" if abs(mi_M - float(ref)) < 1e-6 else "NO coincide: el máximo está donde V = 15 - 6x = 0")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = (PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5 + PRUEBAS_6 + PRUEBAS_7
         + PRUEBAS_8 + PRUEBAS_9)
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")